# Date2Date Seq2Seq：日期格式转换实验

目标：训练一个字符级 seq2seq 模型，将日期字符串从：

`YYYY-MM-DD`

转换为：

`DD/MM/YYYY`

例如：

`2026-06-28 -> 28/06/2026`

本 notebook 当前阶段目标：

1. 搭建最小可运行的 Encoder-Decoder RNN；
2. 跑通训练、生成、评估闭环；
3. 记录 baseline 的失败现象；
4. 分析为什么 vanilla RNN 当前效果差；
5. 为下一轮改进做 checkpoint。

## 1. Imports、随机种子、全局配置

In [ ]:
import torch
import torch.nn as nn
import random

from data import (
    SEED, DEVICE, itos, stoi, SOS_token, EOS_token, vocab_size,
    tensor_from_string, string_from_tensor, make_sample, fixed_samples,
)
from rnn import EncoderRNN, DecoderRNN
from attention_rnn import BahdanauAttention, DecoderRNN_WithAttention
from train import train_batch_samples, generate, eval_samples, char_accuracy


In [ ]:
print("device:", DEVICE)


## 2. 字符表与张量转换工具


In [ ]:
print(itos)
print(stoi)


In [ ]:

x = tensor_from_string("2002-01-23")
print(x, x.shape)
y = tensor_from_string("23/01/2002", add_eos=True)
print(y, y.shape)


In [ ]:
print(string_from_tensor(x))
print(string_from_tensor(y))

3. 数据生成与样本检查

In [ ]:
inputs, targets = make_sample(100)
print(inputs[0:10])
print(targets[0:10])

print("len(inputs)== len(targets)", len(inputs) == len(targets))


In [ ]:
print(fixed_samples)


## 4. Encoder / Decoder 模型定义

本实验先使用最基础的 Encoder-Decoder RNN，不加 attention。

Encoder 将整个输入日期编码成最后一个 hidden state。
Decoder 使用这个 hidden state 作为初始状态，逐字符生成目标日期。

这个 baseline 的限制是：整个输入序列的信息都被压缩到一个 hidden vector 中，
对于日期格式转换这种位置拷贝任务，可能不够稳定。

In [ ]:
hidden_size = 5
encoder = EncoderRNN(vocab_size, hidden_size)

x = tensor_from_string("2002-1-23")
print("x", x.shape)

encoder_output, encoder_hidden = encoder.forward(x);
print("encoder_output:", encoder_output.shape)
print("encoder_hidden:", encoder_hidden.shape)

In [ ]:
input_token = torch.full((1, 1), SOS_token, dtype=torch.long)
decoder = DecoderRNN(vocab_size, hidden_size, vocab_size)
logits, decoder_hidden, pred_token = decoder.forward_step(encoder_hidden, input_token)

print("decoder_hidden:", decoder_hidden.shape)
print("logits:", logits.shape)
print("pred_token", pred_token)


In [ ]:
hidden_size = 5
encoder = EncoderRNN(vocab_size, hidden_size)
decoder = DecoderRNN(vocab_size, hidden_size, vocab_size)

x = tensor_from_string("2002-01-23")
y = tensor_from_string("23/01/2002", add_eos=True)

encoder_output, encoder_hidden = encoder(x)
decoder_outputs, decoder_hidden = decoder(encoder_hidden, y)

print("decoder_outputs:", decoder_outputs.shape)
print("decoder_hidden:", decoder_hidden.shape)
print("target:", y.shape)


## 5. 训练、生成、评估函数

训练、生成、评估函数已分离至 `train.py`，包括：
- `train_batch_samples`：单样本训练
- `generate`：生成（推理）
- `eval_samples`：样本级准确率评估
- `char_accuracy`：字符级准确率评估


## 6. Baseline 实验：当前 vanilla RNN seq2seq

In [ ]:
# 循环训练
EPOCHS = 10
TEST_SIZE = 100
TRAIN_SIZE = 1000
HIDDEN_SIZE = 256
LR = 0.001

loss_group = TRAIN_SIZE//10
epoch_group = EPOCHS//10

inputs, targets = make_sample(TRAIN_SIZE)
samples = list(zip(inputs, targets))
# samples = fixed_samples

encoder = EncoderRNN(vocab_size, HIDDEN_SIZE)
decoder = DecoderRNN(vocab_size, HIDDEN_SIZE,vocab_size)

encoder_optimizer = torch.optim.Adam(encoder.parameters(), LR)
decoder_optimizer = torch.optim.Adam(decoder.parameters(), LR)

criterion = nn.CrossEntropyLoss()
for j in range(EPOCHS):
    encoder.train()
    decoder.train()
    epoch_loss = 0

    random.shuffle(samples)
    for i, (input_str, target_str) in enumerate(samples):
        loss = train_batch_samples(input_str, target_str, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion)
        epoch_loss += loss
    
    if (j + 1) % epoch_group == 0:
        print("EPOCHS:", j + 1, ", avg_epoch_loss:", epoch_loss / len(samples))
        eval_samples(fixed_samples, encoder, decoder, verbose=True)
        

    
            
train_correct, train_total, train_acc = eval_samples(samples, encoder, decoder )
print("train accuracy:", train_correct, "/", len(samples), train_acc)

train_char_accuracy = char_accuracy(samples, encoder, decoder )
print("train char_accuracy:", train_char_accuracy)

test_inputs, test_targets = make_sample(TEST_SIZE)
test_samples = list(zip(test_inputs, test_targets))
test_correct, test_total, test_acc = eval_samples( test_samples, encoder, decoder)
print("test accuracy:", test_correct, "/", len(test_samples), test_acc)
test_char_accuracy = char_accuracy(test_samples, encoder, decoder )
print("test char_accuracy:", test_char_accuracy)

EPOCHS = 10
TEST_SIZE = 100
TRAIN_SIZE = 2000
HIDDEN_SIZE = 256
LR = 0.001

loss_group = TRAIN_SIZE//10
epoch_group = EPOCHS//10

EPOCHS: 10 , avg_epoch_loss: 0.5933430655971169
input:   2002-01-23
pred:    23/02/1964
target:  23/01/2002
correct: False
---
input:   1999-12-08
pred:    08/12/1964
target:  08/12/1999
correct: False
---
input:   2020-03-07
pred:    07/05/1964
target:  07/03/2020
correct: False
---
input:   1950-11-28
pred:    28/11/2022
target:  28/11/1950
correct: False
---
input:   2048-06-15
pred:    15/03/1964
target:  15/06/2048
correct: False
---
train accuracy: 11 / 2000 0.0055
train char_accuracy: 0.65955
test accuracy: 0 / 100 0.0
test char_accuracy: 0.676

dd/mm比较正确，但是yyyy准确率低。准备换一下模型试一下。

**更换成 GRU 后的结果：**

```
EPOCHS: 10 , avg_epoch_loss: 0.033306556500232544
input:   2002-01-23
pred:    23/01/2002
target:  23/01/2002
correct: True
---
input:   1999-12-08
pred:    08/12/1997
target:  08/12/1999
correct: False
---
input:   2020-03-07
pred:    07/03/2020
target:  07/03/2020
correct: True
---
input:   1950-11-28
pred:    28/11/1950
target:  28/11/1950
correct: True
---
input:   2048-06-15
pred:    15/06/2048
target:  15/06/2048
correct: True
---
train accuracy: 923 / 1000 0.923
train char_accuracy: 0.9902
test accuracy: 93 / 100 0.93
test char_accuracy: 0.99
```


7. Bahdanau Attention
使用加性注意力来改造 seq2seq。
将encoder的所有时间步的hidden的最后一层收集起来作为key和value。decoder中时间t-1的hidden作为query。
注意key和value都是encoder的hidden。
Bahdanau Attention 的生成 context 的embedding拼接到一起成为了 decode的新的input，代替了原先只有embedding的情况。

In [ ]:
# 循环训练
EPOCHS = 10
TEST_SIZE = 100
TRAIN_SIZE = 1000
HIDDEN_SIZE = 256
LR = 0.001

loss_group = TRAIN_SIZE//10
epoch_group = EPOCHS//10

inputs, targets = make_sample(TRAIN_SIZE)
samples = list(zip(inputs, targets))
# samples = fixed_samples

encoder = EncoderRNN(vocab_size, HIDDEN_SIZE)
decoder = DecoderRNN_WithAttention(vocab_size, HIDDEN_SIZE,vocab_size)

encoder_optimizer = torch.optim.Adam(encoder.parameters(), LR)
decoder_optimizer = torch.optim.Adam(decoder.parameters(), LR)

criterion = nn.CrossEntropyLoss()
for j in range(EPOCHS):
    encoder.train()
    decoder.train()
    epoch_loss = 0

    random.shuffle(samples)
    for i, (input_str, target_str) in enumerate(samples):
        loss = train_batch_samples(input_str, target_str, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion)
        epoch_loss += loss
    
    if (j + 1) % epoch_group == 0:
        print("EPOCHS:", j + 1, ", avg_epoch_loss:", epoch_loss / len(samples))
        eval_samples(fixed_samples, encoder, decoder, verbose=True)
        

    
            
train_correct, train_total, train_acc = eval_samples(samples, encoder, decoder )
print("train accuracy:", train_correct, "/", len(samples), train_acc)

train_char_accuracy = char_accuracy(samples, encoder, decoder )
print("train char_accuracy:", train_char_accuracy)

test_inputs, test_targets = make_sample(TEST_SIZE)
test_samples = list(zip(test_inputs, test_targets))
test_correct, test_total, test_acc = eval_samples( test_samples, encoder, decoder)
print("test accuracy:", test_correct, "/", len(test_samples), test_acc)
test_char_accuracy = char_accuracy(test_samples, encoder, decoder )
print("test char_accuracy:", test_char_accuracy)